# banana_to_pdf

Converts a [BananaDrum](https://bananadrum.net) shareable link into a printable PDF.

❓ Something not working? See the [FAQ](https://github.com/bruno-ritmista/samba/blob/main/banana_to_pdf/FAQ.md).

In [ ]:
#@title { display-mode: "form" }
#@markdown Before running this converter, write your samba notes in [BananaDrum](https://bananadrum.net), genereate a shareable link and copy it.
#@markdown
#@markdown **Step 1) (mandatory)** Paste the shareable BananaDrum link in the field bananadrum_url below.
#@markdown
#@markdown **Step 2) (optional)** For test purposes only: You may choose to run a test version of the conversion tool by entering the corresponding branch name. **Leave this field empty** to run the standard version.
#@markdown
#@markdown Click ▶ to generate the PDF and download it.

bananadrum_url = "" #@param {type:"string", placeholder:"Paste the BananaDrum link here"}
branch_name = "" #@param {type:"string", placeholder:"Advanced - leave empty to install from main"}

import importlib

# Branch to install banana_to_pdf from. Normal users leave branch_name
# empty and get "main". To test in-progress work, open this notebook from a
# branch's Colab URL and type that branch's name into the branch_name field.
_branch = branch_name.strip() or "main"
if importlib.util.find_spec("banana_to_pdf") is None or globals().get("_installed_branch") != _branch:
    print(f"⌛ Installing banana_to_pdf from branch '{_branch}' — this may take about 30 seconds...")
    import subprocess, shutil
    if shutil.which("uv") is None:
        subprocess.run(["pip", "install", "uv", "-q"], check=True, capture_output=True)
    _install = subprocess.run(
        ["uv", "pip", "install", "--reinstall-package", "banana_to_pdf",
         f"git+https://github.com/bruno-ritmista/samba.git@{_branch}#subdirectory=banana_to_pdf",
         "--system", "-q"],
        capture_output=True
    )
    if _install.returncode != 0:
        print("❌ Installation failed:\n" + _install.stderr.decode())
        raise SystemExit(1)
    _installed_branch = _branch
    print("✅ Conversion tool ready.")

import threading, time
if not any(t.name == "colab_keepalive" for t in threading.enumerate()):
    def _keep_alive():
        while True:
            time.sleep(60)
    _t = threading.Thread(target=_keep_alive, name="colab_keepalive", daemon=True)
    _t.start()

import logging as _logging

def _setup_notebook_logging():
    class _FriendlyFormatter(_logging.Formatter):
        def format(self, record):
            if record.levelno == _logging.WARNING:
                return f"⚠️  {record.getMessage()}"
            return record.getMessage()

    _h = _logging.StreamHandler()
    _h.setFormatter(_FriendlyFormatter())
    _pkg = _logging.getLogger("banana_to_pdf")
    _pkg.handlers.clear()
    _pkg.addHandler(_h)
    _pkg.propagate = False
    _pkg.setLevel(_logging.WARNING)

_setup_notebook_logging()

def _run():
    url = bananadrum_url.strip()

    if not url:
        print("Paste a BananaDrum link in the field above, then run this cell.")
        return

    try:
        from banana_to_pdf.decode import decode_url
        from banana_to_pdf.mapping import map_tracks
        from banana_to_pdf.render import render_pdf
        from banana_to_pdf.__main__ import _default_output_path
    except ImportError:
        print("❌ The conversion tool failed to load. Please reload the page and try again.")
        return

    try:
        decoded = decode_url(url)
    except Exception:
        print("❌ Could not read that link. Please check it was copied from bananadrum.net and try again.")
        return

    rows = map_tracks(decoded)
    if not rows:
        print("❌ No recognised instruments with notes were found in that link.")
        return

    out_path = _default_output_path(decoded.title)
    render_pdf(rows, decoded.n_bars, decoded.title, url, out_path)

    print(f"✅ PDF for \"{decoded.title or 'Untitled'}\" "
          f"({decoded.n_bars} bars, {len(rows)} instruments) is ready:")

    from google.colab import files
    files.download(out_path)

_run()